In [1]:
import pickle
import os
import time
import numpy as np

# add local path to find polychrom
import sys
sys.path.insert(0, str('/home/carlos/Clone/polychrom'))

import polychrom

from polychrom import polymerutils
from polychrom import forces
from polychrom import forcekits
from polychrom.simulation import Simulation
from polychrom.starting_conformations import grow_cubic
from polychrom.hdf5_format import HDF5Reporter, list_URIs, load_URI, load_hdf5_file

import simtk.openmm 
import os 
import shutil


import warnings
import h5py 
import glob

## Defining simulation parameters

In [2]:
# -------defining parameters----------
#  -- basic loop extrusion parameters

folder = "trajectory"

myfile = h5py.File("trajectory/LEFPositions.h5", mode='r')

N = myfile.attrs["N"]
LEFNum = myfile.attrs["LEFNum"]
LEFpositions = myfile["positions"]

Nframes = LEFpositions.shape[0]

    
steps = 750   # MD steps per step of cohesin
stiff = 1
dens = 0.1
box = (N / dens) ** 0.33  # density = 0.1.
data = grow_cubic(N, int(box) - 2)  # creates a compact conformation 
block = 0  # starting block 

# new parameters because some things changed 
saveEveryBlocks = 10   # save every 10 blocks (saving every block is now too much almost)
restartSimulationEveryBlocks = 100

# parameters for smc bonds
smcBondWiggleDist = 0.2
smcBondDist = 0.5

# assertions for easy managing code below 
assert (Nframes % restartSimulationEveryBlocks) == 0 
assert (restartSimulationEveryBlocks % saveEveryBlocks) == 0

savesPerSim = restartSimulationEveryBlocks // saveEveryBlocks
simInitsTotal  = (Nframes) // restartSimulationEveryBlocks

## Bond updating functions (for increased speed)

In [9]:
class bondUpdater(object):

    def __init__(self, LEFpositions):
        """
        :param smcTransObject: smc translocator object to work with
        """
        self.LEFpositions = LEFpositions
        self.curtime  = 0
        self.allBonds = []

    def setParams(self, activeParamDict, inactiveParamDict):
        """
        A method to set parameters for bonds.
        It is a separate method because you may want to have a Simulation object already existing

        :param activeParamDict: a dict (argument:value) of addBond arguments for active bonds
        :param inactiveParamDict:  a dict (argument:value) of addBond arguments for inactive bonds

        """
        self.activeParamDict = activeParamDict
        self.inactiveParamDict = inactiveParamDict


    def setup(self, bondForce,  blocks=100, smcStepsPerBlock=1):
        """
        A method that milks smcTranslocator object
        and creates a set of unique bonds, etc.

        :param bondForce: a bondforce object (new after simulation restart!)
        :param blocks: number of blocks to precalculate
        :param smcStepsPerBlock: number of smcTranslocator steps per block
        :return:
        """


        if len(self.allBonds) != 0:
            raise ValueError("Not all bonds were used; {0} sets left".format(len(self.allBonds)))

        self.bondForce = bondForce

        #precalculating all bonds
        allBonds = []
        
        loaded_positions  = self.LEFpositions[self.curtime : self.curtime+blocks]
        allBonds = [[(int(loaded_positions[i, j, 0]), int(loaded_positions[i, j, 1])) 
                        for j in range(loaded_positions.shape[1])] for i in range(blocks)]

        self.allBonds = allBonds
        self.uniqueBonds = list(set(sum(allBonds, [])))

        #adding forces and getting bond indices
        self.bondInds = []
        self.curBonds = allBonds.pop(0)

        for bond in self.uniqueBonds:
            paramset = self.activeParamDict if (bond in self.curBonds) else self.inactiveParamDict
            ind = bondForce.addBond(bond[0], bond[1], **paramset) # changed from addBond
            self.bondInds.append(ind)
        self.bondToInd = {i:j for i,j in zip(self.uniqueBonds, self.bondInds)}
        
        self.curtime += blocks 
        
        return self.curBonds,[]


    def step(self, context, verbose=False):
        """
        Update the bonds to the next step.
        It sets bonds for you automatically!
        :param context:  context
        :return: (current bonds, previous step bonds); just for reference
        """
        if len(self.allBonds) == 0:
            raise ValueError("No bonds left to run; you should restart simulation and run setup  again")

        pastBonds = self.curBonds
        self.curBonds = self.allBonds.pop(0)  # getting current bonds
        bondsRemove = [i for i in pastBonds if i not in self.curBonds]
        bondsAdd = [i for i in self.curBonds if i not in pastBonds]
        bondsStay = [i for i in pastBonds if i in self.curBonds]
        if verbose:
            print("{0} bonds stay, {1} new bonds, {2} bonds removed".format(len(bondsStay),
                                                                            len(bondsAdd), len(bondsRemove)))
        bondsToChange = bondsAdd + bondsRemove
        bondsIsAdd = [True] * len(bondsAdd) + [False] * len(bondsRemove)
        for bond, isAdd in zip(bondsToChange, bondsIsAdd):
            ind = self.bondToInd[bond]
            paramset = self.activeParamDict if isAdd else self.inactiveParamDict
            self.bondForce.setBondParameters(ind, bond[0], bond[1], **paramset)  # actually updating bonds
        self.bondForce.updateParametersInContext(context)  # now run this to update things in the context
        return self.curBonds, pastBonds

## Run simulation 

In [10]:
milker = bondUpdater(LEFpositions)

reporter = HDF5Reporter(folder=folder, max_data_length=100, overwrite=True, blocks_only=False)

for iteration in range(simInitsTotal):
    
    # simulation parameters are defined below 
    a = Simulation(
            platform="cuda",
            integrator="variableLangevin", 
            error_tol=0.01, 
            GPU = "0", 
            collision_rate=0.03, 
            N = len(data),
            reporters=[reporter],
            PBCbox=[box, box, box],
            precision="mixed")  # timestep not necessary for variableLangevin

    ############################## New code ##############################
    a.set_data(data)  # loads a polymer, puts a center of mass at zero
    
    a.add_force(
        forcekits.polymer_chains(
            a,
            chains=[(0, None, False)],

                # By default the library assumes you have one polymer chain
                # If you want to make it a ring, or more than one chain, use self.setChains
                # self.setChains([(0,50,1),(50,None,0)]) will set a 50-monomer ring and a chain from monomer 50 to the end

            bond_force_func=forces.harmonic_bonds,
            bond_force_kwargs={
                'bondLength':1.0,
                'bondWiggleDistance':0.1, # Bond distance will fluctuate +- 0.05 on average
             },

            angle_force_func=forces.angle_force,
            angle_force_kwargs={
                'k':1.5
                # K is more or less arbitrary, k=4 corresponds to presistence length of 4,
                # k=1.5 is recommended to make polymer realistically flexible; k=8 is very stiff
            },

            nonbonded_force_func=forces.polynomial_repulsive,
            nonbonded_force_kwargs={
                'trunc':1.5, # this will let chains cross sometimes
                'radiusMult':1.05, # this is from old code
                #'trunc':10.0, # this will resolve chain crossings and will not let chain cross anymore
            },

            except_bonds=True,
             
        )
    )
    
    # ------------ initializing milker; adding bonds ---------
    # copied from addBond
    kbond = a.kbondScalingFactor / (smcBondWiggleDist ** 2)
    bondDist = smcBondDist * a.length_scale

    activeParams = {"length":bondDist,"k":kbond}
    inactiveParams = {"length":bondDist, "k":0}
    milker.setParams(activeParams, inactiveParams)
     
    # this step actually puts all bonds in and sets first bonds to be what they should be
    milker.setup(bondForce=a.force_dict['harmonic_bonds'],
                blocks=restartSimulationEveryBlocks)

    # If your simulation does not start, consider using energy minimization below
    if iteration==0:
        a.local_energy_minimization() 
    else:
        a._apply_forces()
    
    for i in range(restartSimulationEveryBlocks):        
        if i % saveEveryBlocks == (saveEveryBlocks - 1):  
            a.do_block(steps=steps)
        else:
            a.integrator.step(steps)  # do steps without getting the positions from the GPU (faster)
        if i < restartSimulationEveryBlocks - 1: 
            curBonds, pastBonds = milker.step(a.context)  # this updates bonds. You can do something with bonds here
    data = a.get_data()  # save data and step, and delete the simulation
    del a
    
    reporter.blocks_only = True  # Write output hdf5-files only for blocks

    time.sleep(0.2)  # wait 200ms for sanity (to let garbage collector do its magic)
    
reporter.dump_data()

INFO:root:Performing local energy minimization
INFO:root:adding force harmonic_bonds 0
INFO:root:adding force angle 1
INFO:root:Using periodic boundary conditions
INFO:root:adding force polynomial_repulsive 2


Exclude neighbouring chain particles from polynomial_repulsive
Number of exceptions: 39999


INFO:root:Particles loaded. Potential energy is 1.910116
INFO:root:before minimization eK=1.505174785375179, eP=1.910115751632275, time=0.0 ps
INFO:root:Particles loaded. Potential energy is 0.143035
INFO:root:after minimization eK=1.505174785375179, eP=0.11100732223781438, time=0.0 ps
INFO:root:block    0 pos[1]=[28.2 13.9 32.2] dr=13.95 t=1324.9ps kin=1.46 pot=1.29 Rg=33.040 SPS=3000 dt=176.8fs dx=47.64pm 
INFO:root:block    1 pos[1]=[34.5 13.2 22.8] dr=9.09 t=2650.9ps kin=1.46 pot=1.28 Rg=34.625 SPS=2972 dt=176.8fs dx=47.76pm 
INFO:root:block    2 pos[1]=[38.1 13.1 21.0] dr=8.63 t=3976.9ps kin=1.46 pot=1.27 Rg=34.944 SPS=2983 dt=176.8fs dx=47.80pm 
INFO:root:block    3 pos[1]=[41.2 11.4 15.1] dr=8.66 t=5302.9ps kin=1.47 pot=1.27 Rg=35.179 SPS=2940 dt=176.8fs dx=47.88pm 
INFO:root:block    4 pos[1]=[21.3 21.1 17.6] dr=8.73 t=6628.9ps kin=1.45 pot=1.28 Rg=35.157 SPS=3046 dt=176.8fs dx=47.53pm 
INFO:root:block    5 pos[1]=[26.7 16.7 17.3] dr=8.75 t=7954.9ps kin=1.46 pot=1.28 Rg=35.258 

Exclude neighbouring chain particles from polynomial_repulsive
Number of exceptions: 39999


INFO:root:Particles loaded. Potential energy is 1.319234
INFO:root:block    0 pos[1]=[12.9 6.5 1.2] dr=8.60 t=1278.4ps kin=1.46 pot=1.27 Rg=35.161 SPS=3222 dt=170.5fs dx=46.01pm 
INFO:root:block    1 pos[1]=[9.9 8.5 0.2] dr=8.53 t=2556.9ps kin=1.47 pot=1.28 Rg=35.406 SPS=3331 dt=170.5fs dx=46.14pm 
INFO:root:block    2 pos[1]=[6.4 6.9 0.5] dr=8.59 t=3835.3ps kin=1.46 pot=1.28 Rg=35.380 SPS=3336 dt=170.5fs dx=45.95pm 
INFO:root:block    3 pos[1]=[3.5 10.0 -0.2] dr=8.47 t=5113.7ps kin=1.46 pot=1.27 Rg=35.239 SPS=3319 dt=170.5fs dx=46.01pm 
INFO:root:block    4 pos[1]=[7.0 7.4 5.1] dr=8.49 t=6392.1ps kin=1.46 pot=1.28 Rg=35.184 SPS=3248 dt=170.5fs dx=46.00pm 
INFO:root:block    5 pos[1]=[6.3 8.2 6.5] dr=8.63 t=7670.5ps kin=1.46 pot=1.28 Rg=35.177 SPS=3341 dt=170.5fs dx=45.95pm 
INFO:root:block    6 pos[1]=[13.0 14.2 7.8] dr=8.66 t=8948.9ps kin=1.47 pot=1.27 Rg=35.176 SPS=3335 dt=170.5fs dx=46.13pm 
INFO:root:block    7 pos[1]=[15.9 19.4 6.2] dr=8.62 t=10227.3ps kin=1.46 pot=1.28 Rg=35.239

...

INFO:root:Particles loaded. Potential energy is 1.305023
INFO:root:block    0 pos[1]=[-22.6 114.0 -23.3] dr=8.57 t=1281.5ps kin=1.46 pot=1.27 Rg=35.411 SPS=3269 dt=170.9fs dx=46.08pm 
INFO:root:block    1 pos[1]=[-27.2 113.7 -24.8] dr=8.51 t=2562.9ps kin=1.45 pot=1.27 Rg=35.329 SPS=3313 dt=170.9fs dx=46.00pm 
INFO:root:block    2 pos[1]=[-29.4 105.1 -30.6] dr=8.51 t=3844.3ps kin=1.46 pot=1.27 Rg=35.239 SPS=3314 dt=170.9fs dx=46.15pm 
INFO:root:block    3 pos[1]=[-23.6 107.3 -26.9] dr=8.52 t=5125.7ps kin=1.46 pot=1.28 Rg=35.344 SPS=3317 dt=170.9fs dx=46.08pm 
INFO:root:block    4 pos[1]=[-9.0 108.8 -17.0] dr=8.54 t=6407.2ps kin=1.46 pot=1.27 Rg=35.346 SPS=3283 dt=170.9fs dx=46.10pm 
INFO:root:block    5 pos[1]=[-12.5 106.9 -17.6] dr=8.70 t=7688.6ps kin=1.46 pot=1.28 Rg=35.319 SPS=3318 dt=170.9fs dx=46.11pm 
INFO:root:block    6 pos[1]=[-14.5 97.8 -17.2] dr=8.59 t=8970.0ps kin=1.46 pot=1.28 Rg=35.155 SPS=3324 dt=170.9fs dx=46.04pm 
INFO:root:block    7 pos[1]=[2.0 101.8 -15.4] dr=8.62 t=

Exclude neighbouring chain particles from polynomial_repulsive
Number of exceptions: 39999


INFO:root:Particles loaded. Potential energy is 1.309110
INFO:root:block    0 pos[1]=[1.9 107.8 -23.0] dr=8.54 t=1278.5ps kin=1.46 pot=1.28 Rg=35.196 SPS=3330 dt=170.5fs dx=45.98pm 
INFO:root:block    1 pos[1]=[9.6 106.1 -29.6] dr=8.51 t=2557.0ps kin=1.45 pot=1.27 Rg=35.173 SPS=3299 dt=170.5fs dx=45.88pm 
INFO:root:block    2 pos[1]=[10.1 110.2 -29.8] dr=8.53 t=3835.5ps kin=1.46 pot=1.27 Rg=35.162 SPS=3271 dt=170.5fs dx=46.02pm 
INFO:root:block    3 pos[1]=[8.3 109.0 -20.0] dr=8.58 t=5114.0ps kin=1.46 pot=1.27 Rg=35.269 SPS=3306 dt=170.5fs dx=46.07pm 
INFO:root:block    4 pos[1]=[12.9 114.0 -23.9] dr=8.46 t=6392.4ps kin=1.46 pot=1.28 Rg=35.220 SPS=3314 dt=170.5fs dx=46.08pm 
INFO:root:block    5 pos[1]=[13.0 109.6 -23.5] dr=8.54 t=7670.9ps kin=1.46 pot=1.27 Rg=35.255 SPS=3267 dt=170.5fs dx=45.93pm 
INFO:root:block    6 pos[1]=[13.8 115.1 -20.0] dr=8.61 t=8949.4ps kin=1.45 pot=1.27 Rg=35.316 SPS=3315 dt=170.5fs dx=45.92pm 
INFO:root:block    7 pos[1]=[12.7 106.8 -10.2] dr=8.52 t=10227.9

Exclude neighbouring chain particles from polynomial_repulsive
Number of exceptions: 39999


INFO:root:Particles loaded. Potential energy is 1.291230
INFO:root:block    0 pos[1]=[19.8 110.3 -14.8] dr=8.35 t=1282.8ps kin=1.45 pot=1.28 Rg=35.332 SPS=3273 dt=171.0fs dx=45.95pm 
INFO:root:block    1 pos[1]=[17.5 116.8 -18.6] dr=8.54 t=2565.5ps kin=1.46 pot=1.27 Rg=35.117 SPS=3300 dt=171.0fs dx=46.17pm 
INFO:root:block    2 pos[1]=[19.8 113.6 -18.3] dr=8.44 t=3848.2ps kin=1.45 pot=1.28 Rg=35.357 SPS=3304 dt=171.0fs dx=46.05pm 
INFO:root:block    3 pos[1]=[21.6 114.8 -13.9] dr=8.40 t=5131.0ps kin=1.46 pot=1.28 Rg=35.382 SPS=3264 dt=171.0fs dx=46.21pm 
INFO:root:block    4 pos[1]=[21.6 114.1 -21.6] dr=8.36 t=6413.7ps kin=1.46 pot=1.27 Rg=35.151 SPS=3270 dt=171.0fs dx=46.15pm 
INFO:root:block    5 pos[1]=[20.2 112.0 -20.2] dr=8.63 t=7696.5ps kin=1.46 pot=1.28 Rg=35.297 SPS=3305 dt=171.0fs dx=46.23pm 
INFO:root:block    6 pos[1]=[19.5 108.2 -22.8] dr=8.60 t=8979.2ps kin=1.46 pot=1.27 Rg=35.464 SPS=3308 dt=171.0fs dx=46.23pm 
INFO:root:block    7 pos[1]=[18.5 112.1 -17.2] dr=8.56 t=1026

In [11]:
os.getcwd()

'/home/carlos/Clone/polychrom/examples/loopExtrusion'